In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/EEG_Emotion_Project'
for sub in ['data_stress', 'data_gameemo', 'models', 'results', 'results/cache']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)

!wget -r -N -c -np -nH --cut-dirs=3 -P "{ROOT}/data_stress" https://physionet.org/files/eegmat/1.0.0/
!ls "{ROOT}/data_stress" | head -20

Mounted at /content/drive
--2026-10-17 07:56:10--  https://physionet.org/files/eegmat/1.0.0/
Resolving physionet.org (physionet.org)... 18.25.8.254
Connecting to physionet.org (physionet.org)|18.25.8.254|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/html]
Saving to: ‘/content/drive/MyDrive/EEG_Emotion_Project/data_stress/index.html’

index.html              [ <=>                ]   9.11K  --.-KB/s    in 0s      

Last-modified header missing -- time-stamps turned off.
2026-10-17 07:56:12 (790 MB/s) - ‘/content/drive/MyDrive/EEG_Emotion_Project/data_stress/index.html’ saved [9326]

Loading robots.txt; please ignore errors.
--2026-10-17 07:56:12--  https://physionet.org/robots.txt
Reusing existing connection to physionet.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 22 [text/plain]
Saving to: ‘/content/drive/MyDrive/EEG_Emotion_Project/data_stress/robots.txt’

robots.txt          100%[===================>]      22  --.-KB/s 

In [ ]:
import os, re
import numpy as np, pandas as pd

ROOT = '/content/drive/MyDrive/EEG_Emotion_Project'
G = f'{ROOT}/data_gameemo'

pat_raw = re.compile(r'S(\d+)G(\d)AllRawChannels\.csv$', re.I)
pat_pre = re.compile(r'S(\d+)G(\d)AllChannels\.csv$', re.I)
raw, pre, dup = {}, {}, []
for r, d, files in os.walk(G):
    if 'Gameplay' in r: continue
    for f in files:
        for pat, store in ((pat_raw, raw), (pat_pre, pre)):
            m = pat.match(f)
            if m:
                key = (int(m.group(1)), int(m.group(2))); p = os.path.join(r, f)
                if key in store:
                    dup.append((key, store[key], p))
                else:
                    store[key] = p

print('raw files:', len(raw), '| preprocessed files:', len(pre), '| duplicates:', len(dup))
for key, a, b in dup[:5]:
    print('DUP', key, os.path.getsize(a), os.path.getsize(b), 'same size' if os.path.getsize(a)==os.path.getsize(b) else 'DIFFERENT')
missing = [(s, g) for s in range(1, 29) for g in range(1, 5) if (s, g) not in raw]
print('missing raw (subject, game):', missing)

# Load every raw file: shape, NaNs, value range
rows = []
for (s, g), p in sorted(raw.items()):
    df = pd.read_csv(p)
    rows.append(dict(s=s, g=g, n=len(df), cols=df.shape[1], nan=int(df.isna().sum().sum()),
                     mean=df.select_dtypes('number').values.mean().round(1),
                     std=df.select_dtypes('number').values.std(0).mean().round(1)))
info = pd.DataFrame(rows)
print('\nColumns:', df.columns.tolist())
print(df.head(3))
print('\nRows per file: min', info.n.min(), 'max', info.n.max(), '| unique counts:', info.n.nunique())
print('Column counts:', info.cols.unique(), '| files with NaNs:', (info.nan > 0).sum())
print(info.describe().round(1))
print('\nShortest 5 files:'); print(info.nsmallest(5, 'n'))

raw files: 108 | preprocessed files: 112 | duplicates: 12
DUP (26, 1) 4341961 4247078 DIFFERENT
DUP (26, 3) 4388467 4273737 DIFFERENT
DUP (26, 2) 4377272 4285620 DIFFERENT
DUP (26, 4) 4386411 4280811 DIFFERENT
DUP (26, 2) 4377272 4377272 same size
missing raw (subject, game): [(26, 1), (26, 2), (26, 3), (26, 4)]

Columns: ['AF3', 'AF4', 'F3', 'F4', 'F7', 'F8', 'FC5', 'FC6', 'O1', 'O2', 'P7', 'P8', 'T7', 'T8', 'Unnamed: 14']
       AF3      AF4       F3       F4        F7         F8     FC5      FC6  \
0 -25.5629 -52.6296  13.5334 -38.5947  101.7501 -2685.1121 -7.5181 -18.5454   
1 -33.0815 -57.1411   5.0126 -48.1187   96.2369 -2700.1121 -5.0126 -25.0618   
2 -32.5799 -62.6543   2.5065 -53.6323  111.2369 -2715.1121  9.9874 -31.5777   

        O1       O2       P7       P8       T7       T8  Unnamed: 14  
0   8.5213   7.5181 -18.5454  37.5925  13.5334  18.5454          NaN  
1   5.0126  14.0345 -12.0300  35.0859  16.0389  12.0291          NaN  
2  15.5383  20.5504  -2.5065  22.5558  20.

In [ ]:
import os
S = f'{ROOT}/data_stress'
for r, d, files in os.walk(S):
    print(r, '|', len(files), 'files |', sorted(files)[:6])
    break   # top level only
print('Total .edf files:', sum(f.endswith('.edf') for _, _, fs in os.walk(S) for f in fs))

/content/drive/MyDrive/EEG_Emotion_Project/data_stress | 78 files | ['README.txt', 'RECORDS', 'SHA256SUMS.txt', 'Subject00_1.edf', 'Subject00_2.edf', 'Subject01_1.edf']
Total .edf files: 72


In [ ]:
# 1) Where is S26, and what is in it?
for r, d, files in os.walk(G):
    if 'S26' in r and 'Gameplay' not in r:
        print(os.path.relpath(r, G), '|', len(files), sorted(files)[:4])

# 2) Every duplicate with full paths
print()
for key, a, b in dup:
    print(key, '\n   A:', os.path.relpath(a, G), os.path.getsize(a),
          '\n   B:', os.path.relpath(b, G), os.path.getsize(b))

# 3) The 16-column raw files
bad = info[info.cols != 15]
print('\n16-col files:', list(zip(bad.s, bad.g)))
for s, g in list(zip(bad.s, bad.g))[:3]:
    print((s, g), pd.read_csv(raw[(s, g)], nrows=2).columns.tolist())

# 4) Stress dataset: README + checksum verification
import hashlib
S = f'{ROOT}/data_stress'
print('\n--- README ---\n', open(f'{S}/README.txt').read()[:1500])
ok = badc = miss = 0
for line in open(f'{S}/SHA256SUMS.txt'):
    parts = line.split()
    if len(parts) < 2: continue
    h, name = parts[0], parts[-1]
    p = os.path.join(S, name)
    if not os.path.exists(p): miss += 1; continue
    if hashlib.sha256(open(p, 'rb').read()).hexdigest() == h: ok += 1
    else: badc += 1
print('checksums OK:', ok, '| bad:', badc, '| missing:', miss)

(S28)/(S26) | 0 []
(S28)/(S26)/Raw EEG Data | 0 []
(S28)/(S26)/Raw EEG Data/.mat format | 4 ['S26G1AllRawChannels.mat', 'S26G2AllRawChannels.mat', 'S26G3AllRawChannels.mat', 'S26G4AllRawChannels.mat']
(S28)/(S26)/Raw EEG Data/.csv format | 4 ['S26G1AllChannels.csv', 'S26G2AllChannels.csv', 'S26G3AllChannels.csv', 'S26G4AllChannels.csv']
(S28)/(S26)/Preprocessed EEG Data | 0 []
(S28)/(S26)/Preprocessed EEG Data/.mat format | 4 ['S26G1AllChannels.mat', 'S26G2AllChannels.mat', 'S26G3AllChannels.mat', 'S26G4AllChannels.mat']
(S28)/(S26)/Preprocessed EEG Data/.csv format | 4 ['S26G1AllChannels.csv', 'S26G2AllChannels.csv', 'S26G3AllChannels.csv', 'S26G4AllChannels.csv']
(S28)/(S26)/SAM Ratings | 4 ['G1.pdf', 'G2.pdf', 'G3.pdf', 'G4.pdf']
(S26) | 0 []
(S26)/Raw EEG Data | 0 []
(S26)/Raw EEG Data/.mat format | 4 ['S26G1AllRawChannels.mat', 'S26G2AllRawChannels.mat', 'S26G3AllRawChannels.mat', 'S26G4AllRawChannels.mat']
(S26)/Raw EEG Data/.csv format | 4 ['S26G1AllChannels.csv', 'S26G2AllChann

In [ ]:
!pip install -q mne
import os, re, hashlib, numpy as np, pandas as pd, mne
ROOT = '/content/drive/MyDrive/EEG_Emotion_Project'
G = f'{ROOT}/data_gameemo'; S = f'{ROOT}/data_stress'
CH = ['AF3','AF4','F3','F4','F7','F8','FC5','FC6','O1','O2','P7','P8','T7','T8']

# ---- GAMEEMO: raw CSVs by folder, skip the stray (S28)/(S26) copy ----
pat = re.compile(r'S(\d+)G(\d)All(Raw)?Channels\.csv$', re.I)
raw = {}
for r, d, files in os.walk(G):
    if 'Raw EEG Data' not in r or not r.endswith('.csv format'): continue
    if os.path.join('(S28)', '(S26)') in r: continue
    for f in files:
        m = pat.match(f)
        if m: raw[(int(m.group(1)), int(m.group(2)))] = os.path.join(r, f)
print('raw recordings found:', len(raw), '| missing:',
      [(s, g) for s in range(1, 29) for g in range(1, 5) if (s, g) not in raw])

X = np.zeros((28, 4, 38252, 14), dtype=np.float32)
for (s, g), p in sorted(raw.items()):
    df = pd.read_csv(p, usecols=CH)[CH]
    assert df.shape == (38252, 14) and not df.isna().any().any(), (s, g, df.shape)
    X[s-1, g-1] = df.values
np.save(f'{ROOT}/results/cache/gameemo_raw.npy', X)

Xc = X - X.mean(2, keepdims=True)
sd = Xc.std(2)                                  # (28, 4, 14) per-channel std in uV
print('channel std (uV): median', np.median(sd).round(1), '| min', sd.min().round(2), '| max', sd.max().round(1))
print('flat channels (std<0.5):', int((sd < 0.5).sum()), '| huge (std>500):', int((sd > 500).sum()))

# ---- Stress: checksums and EDF structure ----
ok = bad = miss = 0
for line in open(f'{S}/SHA256SUMS.txt'):
    parts = line.split()
    if len(parts) < 2: continue
    p = os.path.join(S, parts[-1])
    if not os.path.exists(p): miss += 1; continue
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''): h.update(chunk)
    ok += h.hexdigest() == parts[0]; bad += h.hexdigest() != parts[0]
print('\nchecksums OK:', ok, '| bad:', bad, '| missing:', miss)

for f in ['Subject00_1.edf', 'Subject00_2.edf']:
    e = mne.io.read_raw_edf(f'{S}/{f}', preload=False, verbose=False)
    print(f, '| sfreq', e.info['sfreq'], '| channels', len(e.ch_names), '| seconds', round(e.times[-1], 1))
print(e.ch_names)
print([x for x in os.listdir(S) if not x.endswith('.edf')])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 38.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
raw recordings found: 112 | missing: []
channel std (uV): median 40.9 | min 5.46 | max 393.5
flat channels (std<0.5): 0 | huge (std>500): 0

checksums OK: 75 | bad: 0 | missing: 0
Subject00_1.edf | sfreq 500.0 | channels 21 | seconds 182.0
Subject00_2.edf | sfreq 500.0 | channels 21 | seconds 62.0
['EEG Fp1', 'EEG Fp2', 'EEG F3', 'EEG F4', 'EEG F7', 'EEG F8', 'EEG T3', 'EEG T4', 'EEG C3', 'EEG C4', 'EEG T5', 'EEG T6', 'EEG P3', 'EEG P4', 'EEG O1', 'EEG O2', 'EEG Fz', 'EEG Cz', 'EEG Pz', 'EEG A2-A1', 'ECG ECG']
['README.txt', 'RECORDS', 'subject-info.csv', 'SHA256SUMS.txt', 'index.htm

In [ ]:
import mne
info = pd.read_csv(f'{S}/subject-info.csv')
print(info.columns.tolist()); print(info.head(8)); print(info.iloc[:, 1:].describe(include='all').T.head(8))

names0 = None; rows = []
for s in range(36):
    for k in (1, 2):
        e = mne.io.read_raw_edf(f'{S}/Subject{s:02d}_{k}.edf', preload=False, verbose=False)
        if names0 is None: names0 = e.ch_names
        rows.append(dict(s=s, k=k, fs=e.info['sfreq'], nch=len(e.ch_names),
                         sec=round(e.times[-1], 1), same_names=(e.ch_names == names0)))
inv = pd.DataFrame(rows)
print('\nsampling rates:', inv.fs.unique(), '| channel counts:', inv.nch.unique(),
      '| all channel names identical:', inv.same_names.all())
print(inv.groupby('k').sec.describe().round(1))
print('\nShortest task (k=2) recordings:'); print(inv[inv.k == 2].nsmallest(4, 'sec'))
print('Shortest rest (k=1) recordings:'); print(inv[inv.k == 1].nsmallest(4, 'sec'))

['Subject', 'Age', 'Gender', 'Recording year', 'Number of subtractions', 'Count quality']
     Subject  Age Gender  Recording year  Number of subtractions  \
0  Subject00   21      F            2011                    9.70   
1  Subject01   18      F            2011                   29.35   
2  Subject02   19      F            2012                   12.88   
3  Subject03   17      F            2010                   31.00   
4  Subject04   17      F            2010                    8.60   
5  Subject05   16      F            2010                   20.71   
6  Subject06   18      M            2011                    4.35   
7  Subject07   18      F            2012                   13.38   

   Count quality  
0              0  
1              1  
2              1  
3              1  
4              0  
5              1  
6              0  
7              1  
                       count unique  top freq         mean       std     min  \
Age                     36.0    NaN  NaN  NaN 

In [ ]:
import os, numpy as np, mne, time
mne.set_log_level('ERROR')
ROOT = '/content/drive/MyDrive/EEG_Emotion_Project'; S = f'{ROOT}/data_stress'
EEG = ['Fp1','Fp2','F3','F4','F7','F8','T3','T4','C3','C4','T5','T6','P3','P4','O1','O2','Fz','Cz','Pz']
FS, WIN, STEP, CLIP = 128, 512, 256, 60

def load_clip(s, k):
    r = mne.io.read_raw_edf(f'{S}/Subject{s:02d}_{k}.edf', preload=True, verbose=False)
    names = [f'EEG {c}' for c in EEG]
    r.pick(names); r.reorder_channels(names)
    r.filter(1., 45., verbose=False); r.resample(FS)
    x = r.get_data() * 1e6                              # volts -> microvolts
    return x[:, -CLIP*FS:] if k == 1 else x[:, FS:FS + CLIP*FS]

Xw, y, subj, rec, dropped, t0 = [], [], [], [], 0, time.time()
for s in range(36):
    for k in (1, 2):
        x = load_clip(s, k)
        for a in range(0, x.shape[1] - WIN + 1, STEP):
            w = x[:, a:a + WIN]
            if np.ptp(w, axis=1).max() > 500: dropped += 1; continue   # crude artifact reject
            Xw.append(w); y.append(k - 1); subj.append(s); rec.append(s * 10 + k)
Xw = np.array(Xw, dtype=np.float32); y = np.array(y); subj = np.array(subj); rec = np.array(rec)
np.savez(f'{ROOT}/results/cache/stress_windows.npz', Xw=Xw, y=y, subj=subj, rec=rec)
print(Xw.shape, '| rest:', (y == 0).sum(), 'stress:', (y == 1).sum(), '| dropped:', dropped,
      '| subjects:', len(set(subj)), f'| {time.time()-t0:.0f}s')
print('median channel std (uV):', np.median(Xw.std(-1)).round(1))

(2086, 19, 512) | rest: 1044 stress: 1042 | dropped: 2 | subjects: 36 | 28s
median channel std (uV): 9.1


In [ ]:
from scipy.signal import welch
BANDS = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
L, R = list(range(0, 16, 2)), list(range(1, 16, 2))     # Fp1-Fp2, F3-F4, F7-F8, T3-T4, C3-C4, T5-T6, P3-P4, O1-O2

def features(W):
    f, p = welch(W, fs=FS, nperseg=256, noverlap=128, axis=-1)      # (n, 19, freqs)
    df = f[1] - f[0]
    bp = np.stack([p[..., (f >= lo) & (f < hi)].sum(-1) * df for lo, hi in BANDS.values()], -1)
    lp = np.log(bp + 1e-12)                                          # (n, 19, 5)
    asym = (lp[:, L] - lp[:, R]).reshape(len(W), -1)                 # 8 pairs x 5 = 40
    ratio = lp[:, :, 2] - lp[:, :, 3]                                # log(alpha/beta), 19
    return np.concatenate([lp.reshape(len(W), -1), asym, ratio], 1)  # 95 + 40 + 19 = 154

F = features(Xw)
for s in np.unique(subj):                                            # per-subject z-score
    m = subj == s
    F[m] = (F[m] - F[m].mean(0)) / (F[m].std(0) + 1e-8)
F = F.astype(np.float32)
np.savez(f'{ROOT}/results/cache/stress_features.npz', F=F, y=y, subj=subj, rec=rec)
print('features:', F.shape, '| NaNs:', int(np.isnan(F).sum()))

features: (2086, 154) | NaNs: 0


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score

def oof(make, yy):
    p = np.zeros(len(yy))
    for tr, te in GroupKFold(6).split(F, yy, subj):                  # whole subjects held out
        p[te] = make().fit(F[tr], yy[tr]).predict_proba(F[te])[:, 1]
    return p

def report(name, p, yy):
    w = (roc_auc_score(yy, p), f1_score(yy, p > .5, average='macro'), balanced_accuracy_score(yy, p > .5))
    ur = np.unique(rec); pr = np.array([p[rec == r].mean() for r in ur]); yr = np.array([yy[rec == r][0] for r in ur])
    r_ = (roc_auc_score(yr, pr), f1_score(yr, pr > .5, average='macro'), balanced_accuracy_score(yr, pr > .5))
    print(f'{name:4s} window-level AUC/F1/BalAcc = {np.round(w, 3)} | recording-level (72 recs) = {np.round(r_, 3)}')

lr = lambda: LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000)
rf = lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight='balanced',
                                    n_jobs=-1, random_state=42)
p_lr = oof(lr, y); report('LR', p_lr, y)
p_rf = oof(rf, y); report('RF', p_rf, y)

LR   window-level AUC/F1/BalAcc = [0.851 0.767 0.767] | recording-level (72 recs) = [0.981 0.903 0.903]
RF   window-level AUC/F1/BalAcc = [0.889 0.814 0.814] | recording-level (72 recs) = [0.981 0.931 0.931]


In [ ]:
# (a) Shuffled labels should give AUC ~0.5. If not, something leaks.
rng = np.random.RandomState(0)
ur = np.unique(rec); perm = dict(zip(ur, rng.permutation([y[rec == r][0] for r in ur])))
y_shuf = np.array([perm[r] for r in rec])
report('LRsh', oof(lr, y_shuf), y_shuf)

# (b) Does performance differ for good vs poor counters?
import pandas as pd
ci = pd.read_csv(f'{S}/subject-info.csv')['Count quality'].values   # row i = Subject i
good = ci[subj] == 1
for name, m in [('good counters', good), ('poor counters', ~good)]:
    print(f'{name:14s} window AUC =', round(roc_auc_score(y[m], p_lr[m]), 3))

# (c) Which features drive it? (top LR coefficients, full data)
names = [f'{c}_{b}' for c in EEG for b in BANDS] + [f'asym{i}_{b}' for i in range(8) for b in BANDS] + [f'ab_{c}' for c in EEG]
coef = lr().fit(F, y).coef_[0]
for i in np.argsort(-np.abs(coef))[:8]: print(f'{names[i]:14s} {coef[i]:+.2f}')

LRsh window-level AUC/F1/BalAcc = [0.517 0.507 0.507] | recording-level (72 recs) = [0.529 0.471 0.472]
good counters  window AUC = 0.85
poor counters  window AUC = 0.852
Fp2_gamma      +1.17
F4_beta        -0.89
O1_theta       -0.64
T4_delta       -0.59
asym3_gamma    -0.58
T3_alpha       -0.57
Fp1_gamma      +0.54
T4_beta        +0.51


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

F_raw = features(Xw)                                   # un-normalised log features
gamma = np.zeros(154, bool); gamma[[c*5+4 for c in range(19)] + [95+i*5+4 for i in range(8)]] = True

def rest_baseline(Fr):                                 # z-score using ONLY each subject's rest windows
    out = Fr.copy()
    for s in np.unique(subj):
        m = subj == s; b = m & (y == 0)
        out[m] = (Fr[m] - Fr[b].mean(0)) / (Fr[b].std(0) + 1e-8)
    return out

def oof2(make, Fm, yy):
    p = np.zeros(len(yy))
    for tr, te in GroupKFold(6).split(Fm, yy, subj):
        p[te] = make().fit(Fm[tr], yy[tr]).predict_proba(Fm[te])[:, 1]
    return p

lr_s = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000))
FB = rest_baseline(F_raw)
for name, Fm, mk in [('A rest-baseline only      ', FB, rf),
                     ('B no normalisation        ', F_raw, rf),
                     ('C rest-baseline, no gamma ', FB[:, ~gamma], rf),
                     ('D no normalisation, no gamma', F_raw[:, ~gamma], rf)]:
    p = oof2(mk, Fm, y)
    ur = np.unique(rec); pr = np.array([p[rec == r].mean() for r in ur]); yr = np.array([y[rec == r][0] for r in ur])
    print(f'{name} window AUC {roc_auc_score(y, p):.3f} BalAcc {balanced_accuracy_score(y, p > .5):.3f} '
          f'| recording AUC {roc_auc_score(yr, pr):.3f} BalAcc {balanced_accuracy_score(yr, pr > .5):.3f}')

A rest-baseline only       window AUC 0.944 BalAcc 0.871 | recording AUC 0.969 BalAcc 0.958
B no normalisation         window AUC 0.748 BalAcc 0.657 | recording AUC 0.789 BalAcc 0.681
C rest-baseline, no gamma  window AUC 0.912 BalAcc 0.817 | recording AUC 0.995 BalAcc 0.903
D no normalisation, no gamma window AUC 0.719 BalAcc 0.641 | recording AUC 0.782 BalAcc 0.694


In [ ]:
# ---------- 1) Check the GAMEEMO label mapping on ALL subjects ----------
!pip install -q pypdf
import os, re, numpy as np, pandas as pd
from pypdf import PdfReader
G = f'{ROOT}/data_gameemo'
rat = {}
for r, d, files in os.walk(G):
    if 'SAM Ratings' not in r: continue
    sid = [int(x) for x in re.findall(r'\(S(\d+)\)', r)][-1]
    for f in files:
        if not re.match(r'G[1-4]\.pdf$', f): continue
        t = PdfReader(os.path.join(r, f)).pages[0].extract_text()
        nums = [int(l) for l in t.splitlines() if l.strip().isdigit()]
        if len(nums) >= 5: rat[(sid, int(f[1]))] = nums[-5:]   # satisfied, boring, horrible, calm, funny
print('forms parsed:', len(rat), 'of 112')
R = pd.DataFrame([dict(s=s, g=g, satisfied=v[0], boring=v[1], horrible=v[2], calm=v[3], funny=v[4])
                  for (s, g), v in rat.items()])
M = R.groupby('g')[['boring', 'horrible', 'calm', 'funny']].mean().round(2)
print(M); print('game with highest mean rating per word:', M.idxmax().to_dict())

# ---------- 2) Stricter stress evaluation (baseline from first part of rest only) ----------
j = np.zeros(len(y), int)
for r_ in np.unique(rec):
    idx = np.where(rec == r_)[0]; j[idx] = np.arange(len(idx))
is_base = (y == 0) & (j < 14)                     # first ~30 s of rest -> baseline only
is_gap  = (y == 0) & (j >= 14) & (j < 16)         # gap so baseline and test windows don't overlap
FB2 = F_raw.copy()
for s in np.unique(subj):
    m = subj == s; b = m & is_base
    FB2[m] = (F_raw[m] - F_raw[b].mean(0)) / (F_raw[b].std(0) + 1e-8)
keep = ~(is_base | is_gap)
sk, yk, rk = subj[keep], y[keep], rec[keep]
print('\nwindows kept:', keep.sum(), '| rest:', (yk == 0).sum(), 'stress:', (yk == 1).sum())

for name, Fm in [('all features', FB2[keep]), ('no gamma   ', FB2[keep][:, ~gamma])]:
    p = np.zeros(len(yk))
    for tr, te in GroupKFold(6).split(Fm, yk, sk):
        p[te] = rf().fit(Fm[tr], yk[tr]).predict_proba(Fm[te])[:, 1]
    ur = np.unique(rk); pr = np.array([p[rk == r_].mean() for r_ in ur]); yr = np.array([yk[rk == r_][0] for r_ in ur])
    print(f'{name} window AUC {roc_auc_score(yk, p):.3f} BalAcc {balanced_accuracy_score(yk, p > .5):.3f} '
          f'| recording AUC {roc_auc_score(yr, pr):.3f} BalAcc {balanced_accuracy_score(yr, pr > .5):.3f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 6.1 MB/s eta 0:00:00


forms parsed: 112 of 112
   boring  horrible  calm  funny
g                               
1    7.14      5.54  1.61   2.79
2    2.43      5.75  2.61   6.07
3    1.50      4.14  6.46   2.11
4    1.14      3.39  1.89   7.96
game with highest mean rating per word: {'boring': 1, 'horrible': 2, 'calm': 3, 'funny': 4}

windows kept: 1510 | rest: 468 stress: 1042
all features window AUC 0.878 BalAcc 0.796 | recording AUC 0.942 BalAcc 0.889
no gamma    window AUC 0.824 BalAcc 0.742 | recording AUC 0.916 BalAcc 0.847


In [ ]:
import logging, os, re, numpy as np, pandas as pd
from collections import Counter
from scipy.optimize import linear_sum_assignment
from pypdf import PdfReader
logging.getLogger('pypdf').setLevel(logging.ERROR)
WORDS = ['boring', 'horrible', 'calm', 'funny']

# rebuild the table with file paths
paths = {}
for r, d, files in os.walk(G):
    if 'SAM Ratings' not in r: continue
    sid = [int(x) for x in re.findall(r'\(S(\d+)\)', r)][-1]
    for f in files:
        if re.match(r'G[1-4]\.pdf$', f): paths[(sid, int(f[1]))] = os.path.join(r, f)

# 1) Raw text of one form that is NOT S01, so we can see the parser's input
k = (5, 2)
print('RAW TEXT for', k, '\n', PdfReader(paths[k]).pages[0].extract_text()[-260:], '\n')

# 2) Per-subject best matching of game files to words
perm_count, quality, rows = Counter(), [], []
for s in range(1, 29):
    Mx = np.array([[R[(R.s == s) & (R.g == g)][w].values[0] for w in WORDS] for g in range(1, 5)], float)
    ri, ci = linear_sum_assignment(-Mx)                      # each word assigned to exactly one game
    perm = tuple(WORDS[c] for c in ci)                       # word that best fits G1, G2, G3, G4
    perm_count[perm] += 1; quality.append(Mx[ri, ci].mean())
    rows.append((s, perm, round(Mx[ri, ci].mean(), 1)))
print('Patterns (word best fitting G1, G2, G3, G4) and number of subjects:')
for p, n in perm_count.most_common(): print(' ', n, p)
print('mean rating of the matched word, per subject (high = clean match):')
print(' ', [q for _, _, q in rows])
print('worst-matching subjects:', sorted(rows, key=lambda x: x[2])[:5])
print('\nS01-like pattern (boring, calm, horrible, funny):', perm_count[('boring', 'calm', 'horrible', 'funny')])
print('Population-mean pattern (boring, horrible, calm, funny):', perm_count[('boring', 'horrible', 'calm', 'funny')])

RAW TEXT for (5, 2) 
 ………………………………………………………...................
....................................................................................................................................................... 
 
SELF-ASSESSMENT MANIKIN 
 
 
Female
25
S05 (Subject 5)
8
2
1
5
7 

Patterns (word best fitting G1, G2, G3, G4) and number of subjects:
  17 ('boring', 'horrible', 'calm', 'funny')
  6 ('boring', 'calm', 'horrible', 'funny')
  3 ('horrible', 'boring', 'calm', 'funny')
  2 ('boring', 'funny', 'calm', 'horrible')
mean rating of the matched word, per subject (high = clean match):
  [np.float64(6.5), np.float64(8.0), np.float64(8.5), np.float64(8.8), np.float64(7.0), np.float64(7.5), np.float64(6.0), np.float64(7.2), np.float64(8.8), np.float64(8.2), np.float64(7.8), np.float64(7.0), np.float64(7.5), np.float64(8.0), np.float64(8.0), np.float64(8.0), np.float64(6.8), np.float64(6.0), np.float64(7.5), np.float64(8.2), np.float64(8.5), np.float64(7.8), np.float64(8.2), np.float6

GAMEEMO

In [ ]:
import numpy as np, pandas as pd, time
from scipy.signal import butter, sosfiltfilt, welch
from scipy.optimize import linear_sum_assignment
from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

Xg = np.load(f'{ROOT}/results/cache/gameemo_raw.npy')            # (28, 4, 38252, 14)
WORDS = ['boring', 'calm', 'horrible', 'funny']                  # class ids 0..3
CLS = {'boring': 'low arousal / low valence', 'calm': 'low arousal / high valence',
       'horrible': 'HIGH arousal / low valence (stress-fear)', 'funny': 'high arousal / high valence'}

# ---- Step 3b: per-subject labels from their own ratings ----
lab = np.zeros((28, 4), int); matched = []
for s in range(1, 29):
    Mx = np.array([[R[(R.s == s) & (R.g == g)][w].values[0] for w in WORDS] for g in range(1, 5)], float)
    ri, ci = linear_sum_assignment(-Mx); lab[s-1] = ci; matched.append(Mx[ri, ci].mean())
print('recordings per class:', np.bincount(lab.ravel()), '| worst subject match:', np.round(min(matched), 1))

# ---- Step 4: preprocess + segment ----
FS, WIN, STEP = 128, 512, 256
sos = butter(4, [1, 45], btype='bandpass', fs=FS, output='sos')
Xw, Gy, Gsubj, Grec, rej = [], [], [], [], 0
for s in range(28):
    for g in range(4):
        x = Xg[s, g].T.astype(np.float64); x -= x.mean(1, keepdims=True)
        x = sosfiltfilt(sos, x, axis=1)[:, 5*FS:]                 # drop first 5 s
        for a in range(0, x.shape[1] - WIN + 1, STEP):
            w = x[:, a:a + WIN]
            if np.ptp(w, axis=1).max() > 400: rej += 1; continue  # crude artifact reject
            Xw.append(w.astype(np.float32)); Gy.append(lab[s, g]); Gsubj.append(s); Grec.append(s*10 + g)
Xw = np.array(Xw); Gy = np.array(Gy); Gsubj = np.array(Gsubj); Grec = np.array(Grec)
print('windows:', Xw.shape, '| rejected:', rej, '| per class:', np.bincount(Gy))

# ---- Step 5: features (14 ch x 5 bands + 7 asymmetry pairs x 5 + alpha/beta ratio) ----
BANDS = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
Lc, Rc = list(range(0, 14, 2)), list(range(1, 14, 2))            # AF3-AF4, F3-F4, F7-F8, FC5-FC6, O1-O2, P7-P8, T7-T8
def gfeat(W):
    f, p = welch(W, fs=FS, nperseg=256, noverlap=128, axis=-1); df = f[1] - f[0]
    bp = np.stack([p[..., (f >= lo) & (f < hi)].sum(-1) * df for lo, hi in BANDS.values()], -1)
    lp = np.log(bp + 1e-12)
    return np.concatenate([lp.reshape(len(W), -1), (lp[:, Lc] - lp[:, Rc]).reshape(len(W), -1), lp[:, :, 2] - lp[:, :, 3]], 1)
GX = np.concatenate([gfeat(Xw[i:i+2000]) for i in range(0, len(Xw), 2000)])
for s in range(28):                                               # per-subject z-score
    m = Gsubj == s; GX[m] = (GX[m] - GX[m].mean(0)) / (GX[m].std(0) + 1e-8)
GX = GX.astype(np.float32); print('features:', GX.shape, '| NaNs:', int(np.isnan(GX).sum()))
np.savez(f'{ROOT}/results/cache/gameemo_features.npz', GX=GX, Gy=Gy, Gsubj=Gsubj, Grec=Grec)

# ---- Step 6: leave-subjects-out evaluation ----
rf4 = lambda: RandomForestClassifier(n_estimators=150, min_samples_leaf=5, max_samples=0.5,
                                     class_weight='balanced', n_jobs=-1, random_state=42)
lr4 = lambda: make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000))
def cv4(make, labels):
    P = np.zeros((len(labels), 4))
    for tr, te in GroupKFold(7).split(GX, labels, Gsubj):
        P[te] = make().fit(GX[tr], labels[tr]).predict_proba(GX[te])
    return P
def m4(P, yy):
    return (f1_score(yy, P.argmax(1), average='macro'), balanced_accuracy_score(yy, P.argmax(1)),
            roc_auc_score(yy, P, multi_class='ovr'))
def report(name, P, yy):
    ur = np.unique(Grec); Pr = np.array([P[Grec == r].mean(0) for r in ur]); yr = np.array([yy[Grec == r][0] for r in ur])
    ar = lambda Q, y_: roc_auc_score(np.isin(y_, [2, 3]), Q[:, 2] + Q[:, 3])
    va = lambda Q, y_: roc_auc_score(np.isin(y_, [1, 3]), Q[:, 1] + Q[:, 3])
    print(f'{name:22s} window F1/BalAcc/AUC {np.round(m4(P, yy), 3)} | recording (112) {np.round(m4(Pr, yr), 3)} '
          f'| arousal AUC {ar(P, yy):.3f}/{ar(Pr, yr):.3f}  valence AUC {va(P, yy):.3f}/{va(Pr, yr):.3f}')
t0 = time.time()
P_lr = cv4(lr4, Gy); report('LR', P_lr, Gy); print(f'({time.time()-t0:.0f}s)', flush=True)
P_rf = cv4(rf4, Gy); report('RF', P_rf, Gy); print(f'({time.time()-t0:.0f}s)')

# ---- Step 7: sanity checks (chance = F1/BalAcc 0.25, AUC 0.5) ----
rng = np.random.RandomState(0)
perm = np.array([rng.permutation(lab[s]) for s in range(28)])                    # shuffle labels within each subject
Gy_shuf = np.array([perm[s, g] for s, g in zip(Gsubj, Grec % 10)])
report('LR shuffled labels', cv4(lr4, Gy_shuf), Gy_shuf)
fixed = np.array([0, 2, 1, 3])                                                   # naive: file number -> majority order
Gy_naive = np.array([fixed[g] for g in Grec % 10])
print('naive fixed mapping agrees with rating-based labels on', round((Gy_naive == Gy).mean(), 3), 'of windows')
report('LR naive file-number', cv4(lr4, Gy_naive), Gy_naive)

recordings per class: [28 28 28 28] | worst subject match: 6.0
windows: (15790, 14, 512) | rejected: 450 | per class: [3962 3968 3910 3950]
features: (15790, 119) | NaNs: 0
LR                     window F1/BalAcc/AUC [0.413 0.413 0.666] | recording (112) [0.581 0.58  0.763] | arousal AUC 0.637/0.728  valence AUC 0.631/0.726
(34s)
RF                     window F1/BalAcc/AUC [0.528 0.531 0.743] | recording (112) [0.656 0.661 0.791] | arousal AUC 0.683/0.711  valence AUC 0.711/0.753
(132s)
LR shuffled labels     window F1/BalAcc/AUC [0.265 0.265 0.508] | recording (112) [0.323 0.321 0.508] | arousal AUC 0.458/0.427  valence AUC 0.500/0.495
naive fixed mapping agrees with rating-based labels on 0.801 of windows
LR naive file-number   window F1/BalAcc/AUC [0.568 0.568 0.803] | recording (112) [0.946 0.946 0.989] | arousal AUC 0.755/0.984  valence AUC 0.801/0.997


In [ ]:
fixed = np.array([0, 2, 1, 3])                    # slot G1..G4 -> class (boring, horrible, calm, funny order of WORDS ids)
slot = Grec % 10
maj = np.array([(lab[s] == fixed).all() for s in range(28)])
tr, te = maj[Gsubj], ~maj[Gsubj]
print('majority-order subjects (train):', maj.sum(), '| deviating subjects (test):', (~maj).sum())

for name, mk in [('LR', lr4), ('RF', rf4)]:
    Pm = mk().fit(GX[tr], Gy[tr]).predict_proba(GX[te])
    gr = Grec[te]; hit_slot = hit_rate = other = 0; n = 0
    for r in np.unique(gr):
        s, g = r // 10, r % 10
        rating, byslot = lab[s, g], fixed[g]
        if rating == byslot: continue              # only the recordings where the two labelings disagree
        pred = Pm[gr == r].mean(0).argmax(); n += 1
        hit_slot += pred == byslot; hit_rate += pred == rating; other += (pred != byslot) and (pred != rating)
    print(f'{name}: {n} disagreeing recordings -> EEG follows FILE SLOT {hit_slot}, RATING-BASED GAME {hit_rate}, neither {other}')

majority-order subjects (train): 17 | deviating subjects (test): 11
LR: 22 disagreeing recordings -> EEG follows FILE SLOT 16, RATING-BASED GAME 1, neither 5
RF: 22 disagreeing recordings -> EEG follows FILE SLOT 17, RATING-BASED GAME 3, neither 2


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
print('GPU:', tf.config.list_physical_devices('GPU'))

# ---- Correct labels: slot = game. 0 boring, 1 calm, 2 horror, 3 funny ----
Gy = (Grec % 10).astype(int)
print('windows per class:', np.bincount(Gy))

# ---- Re-evaluate LR / RF with the right labels and the right arousal/valence grouping ----
t0 = time.time()
P_lr = cv4(lr4, Gy); report('LR', P_lr, Gy)
P_rf = cv4(rf4, Gy); report('RF', P_rf, Gy); print(f'({time.time()-t0:.0f}s)', flush=True)
Gy_shuf = np.concatenate([np.random.RandomState(s).permutation(4)[Gy[Gsubj == s]] for s in range(28)])[np.argsort(np.concatenate([np.where(Gsubj == s)[0] for s in range(28)]))]
report('LR shuffled labels', cv4(lr4, Gy_shuf), Gy_shuf)

# ---- 1D-CNN on raw windows, leave-subjects-out (4 folds) ----
Xn = Xw.transpose(0, 2, 1).astype(np.float32).copy()                   # (n, 512, 14)
for s in range(28):                                                     # per-subject, per-channel z-score
    m = Gsubj == s
    Xn[m] = (Xn[m] - Xn[m].mean((0, 1))) / (Xn[m].std((0, 1)) + 1e-8)

def build_cnn(n_out=4):
    m = models.Sequential([
        layers.Input((512, 14)),
        layers.Conv1D(32, 7, padding='same'), layers.BatchNormalization(), layers.Activation('elu'),
        layers.MaxPool1D(4), layers.Dropout(0.3),
        layers.Conv1D(64, 5, padding='same'), layers.BatchNormalization(), layers.Activation('elu'),
        layers.MaxPool1D(4), layers.Dropout(0.3),
        layers.Conv1D(128, 3, padding='same'), layers.BatchNormalization(), layers.Activation('elu'),
        layers.GlobalAveragePooling1D(),
        layers.Dense(64, activation='elu'), layers.Dropout(0.5), layers.Dense(n_out, activation='softmax')])
    m.compile('adam', 'sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

P_cnn = np.zeros((len(Gy), 4)); t0 = time.time()
for k, (tr, te) in enumerate(GroupKFold(4).split(Xn, Gy, Gsubj)):
    subs = np.unique(Gsubj[tr]); vs = np.random.RandomState(k).choice(subs, 3, replace=False)
    va = np.isin(Gsubj, vs) & np.isin(np.arange(len(Gy)), tr); trn = np.isin(np.arange(len(Gy)), tr) & ~va
    m = build_cnn()
    m.fit(Xn[trn], Gy[trn], validation_data=(Xn[va], Gy[va]), epochs=30, batch_size=128, verbose=0,
          callbacks=[callbacks.EarlyStopping('val_accuracy', patience=6, restore_best_weights=True),
                     callbacks.ReduceLROnPlateau('val_accuracy', factor=0.5, patience=3)])
    P_cnn[te] = m.predict(Xn[te], verbose=0)
    print(f'fold {k+1}/4 done ({time.time()-t0:.0f}s) | fold window acc {(P_cnn[te].argmax(1) == Gy[te]).mean():.3f}', flush=True)
report('1D-CNN', P_cnn, Gy)
report('RF+CNN average', (P_rf + P_cnn) / 2, Gy)

GPU: []
windows per class: [3949 3939 3957 3945]
LR                     window F1/BalAcc/AUC [0.568 0.568 0.803] | recording (112) [0.946 0.946 0.989] | arousal AUC 0.801/0.997  valence AUC 0.755/0.984
RF                     window F1/BalAcc/AUC [0.755 0.755 0.927] | recording (112) [0.973 0.973 1.   ] | arousal AUC 0.905/1.000  valence AUC 0.913/1.000
(135s)
LR shuffled labels     window F1/BalAcc/AUC [0.23  0.23  0.467] | recording (112) [0.215 0.214 0.45 ] | arousal AUC 0.532/0.557  valence AUC 0.397/0.326
fold 1/4 done (531s) | fold window acc 0.710
fold 2/4 done (752s) | fold window acc 0.679
fold 3/4 done (884s) | fold window acc 0.349
fold 4/4 done (1171s) | fold window acc 0.707
1D-CNN                 window F1/BalAcc/AUC [0.609 0.611 0.825] | recording (112) [0.706 0.705 0.881] | arousal AUC 0.833/0.897  valence AUC 0.807/0.892
RF+CNN average         window F1/BalAcc/AUC [0.686 0.687 0.908] | recording (112) [0.792 0.795 0.972] | arousal AUC 0.895/0.960  valence AUC 0.876/0.96

In [ ]:
import joblib
# un-normalised log features (population-level scaling only)
GXraw = np.concatenate([gfeat(Xw[i:i+2000]) for i in range(0, len(Xw), 2000)]).astype(np.float32)
gam = np.zeros(119, bool); gam[[c*5+4 for c in range(14)] + [70+i*5+4 for i in range(7)]] = True
def cvx(Fm, mk=rf4):
    P = np.zeros((len(Gy), 4))
    for tr, te in GroupKFold(7).split(Fm, Gy, Gsubj):
        P[te] = mk().fit(Fm[tr], Gy[tr]).predict_proba(Fm[te])
    return P
t0 = time.time()
for name, Fm in [('B: no per-person normalisation   ', GXraw),
                 ('C: per-person z-score, no gamma  ', GX[:, ~gam]),
                 ('D: no normalisation, no gamma    ', GXraw[:, ~gam])]:
    report(name, cvx(Fm), Gy); print(f'({time.time()-t0:.0f}s)', flush=True)

# ---- save final models (trained on all 28 subjects) ----
MD = f'{ROOT}/models'
joblib.dump(rf4().set_params(n_estimators=300).fit(GX, Gy), f'{MD}/emotion_rf_subjnorm.joblib')
joblib.dump(rf4().set_params(n_estimators=300).fit(GXraw, Gy), f'{MD}/emotion_rf_nonorm.joblib')
Fs = np.load(f'{ROOT}/results/cache/stress_features.npz')
joblib.dump(rf().fit(Fs['F'], Fs['y']), f'{MD}/stress_rf.joblib')
print('saved:', os.listdir(MD))

B: no per-person normalisation    window F1/BalAcc/AUC [0.5   0.502 0.759] | recording (112) [0.564 0.571 0.827] | arousal AUC 0.746/0.806  valence AUC 0.749/0.822
(126s)
C: per-person z-score, no gamma   window F1/BalAcc/AUC [0.741 0.741 0.919] | recording (112) [1. 1. 1.] | arousal AUC 0.896/1.000  valence AUC 0.906/1.000
(214s)
D: no normalisation, no gamma     window F1/BalAcc/AUC [0.494 0.497 0.758] | recording (112) [0.602 0.607 0.843] | arousal AUC 0.753/0.827  valence AUC 0.734/0.809
(319s)
saved: ['emotion_rf_subjnorm.joblib', 'emotion_rf_nonorm.joblib', 'stress_rf.joblib']


In [ ]:
import joblib, os
MD = f'{ROOT}/models'
joblib.dump(rf().fit(FB2, y), f'{MD}/stress_rf_cal.joblib')
if os.path.exists(f'{MD}/stress_rf.joblib'): os.remove(f'{MD}/stress_rf.joblib')   # old, mismatched model
print(os.listdir(MD))

['emotion_rf_subjnorm.joblib', 'emotion_rf_nonorm.joblib', 'stress_rf_cal.joblib']


In [ ]:
%%writefile /content/drive/MyDrive/EEG_Emotion_Project/common.py
import numpy as np
from scipy.signal import butter, sosfiltfilt, welch

FS, WIN, STEP = 128, 512, 256
BANDS = {'delta': (1, 4), 'theta': (4, 8), 'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 45)}
G_CH = ['AF3','AF4','F3','F4','F7','F8','FC5','FC6','O1','O2','P7','P8','T7','T8']
S_CH = ['Fp1','Fp2','F3','F4','F7','F8','T3','T4','C3','C4','T5','T6','P3','P4','O1','O2','Fz','Cz','Pz']
CLASSES = ['Boring', 'Calm', 'Horror (stress/fear)', 'Funny']
_sos = butter(4, [1, 45], btype='bandpass', fs=FS, output='sos')

def feats(W, n_pairs):
    """W: (n, ch, 512) -> features (n, F) and linear band power (n, ch, 5)"""
    f, p = welch(W, fs=FS, nperseg=256, noverlap=128, axis=-1); df = f[1] - f[0]
    bp = np.stack([p[..., (f >= lo) & (f < hi)].sum(-1) * df for lo, hi in BANDS.values()], -1)
    lp = np.log(bp + 1e-12); n = len(W)
    L, R = list(range(0, 2 * n_pairs, 2)), list(range(1, 2 * n_pairs, 2))
    F = np.concatenate([lp.reshape(n, -1), (lp[:, L] - lp[:, R]).reshape(n, -1), lp[:, :, 2] - lp[:, :, 3]], 1)
    return F.astype(np.float32), bp

def windows(x, reject):
    out = [x[:, a:a + WIN] for a in range(0, x.shape[1] - WIN + 1, STEP)
           if np.ptp(x[:, a:a + WIN], axis=1).max() <= reject]
    return np.array(out, dtype=np.float32)

def emotion_windows(df):
    x = df[G_CH].values.T.astype(np.float64); x -= x.mean(1, keepdims=True)
    return windows(sosfiltfilt(_sos, x, axis=1)[:, 5 * FS:], 400)

def stress_clip(path, kind):
    import mne
    r = mne.io.read_raw_edf(path, preload=True, verbose=False)
    names = [f'EEG {c}' for c in S_CH]; r.pick(names); r.reorder_channels(names)
    r.filter(1., 45., verbose=False); r.resample(FS)
    x = r.get_data() * 1e6
    return x[:, -60 * FS:] if kind == 'rest' else x[:, FS:FS + 60 * FS]

Overwriting /content/drive/MyDrive/EEG_Emotion_Project/common.py


In [ ]:
import importlib, sys; sys.path.append(ROOT)
import common; importlib.reload(common)
W = common.emotion_windows(pd.read_csv(raw[(1, 1)]))
Fx, _ = common.feats(W, 7); m = (Gsubj == 0) & (Grec % 10 == 0)
print('emotion:', Fx.shape, GXraw[m].shape, 'match:', np.allclose(Fx, GXraw[m], atol=1e-3))
Wr = common.windows(common.stress_clip(f'{S}/Subject00_1.edf', 'rest'), 500)
Fr, _ = common.feats(Wr, 8); m = (subj == 0) & (y == 0)
print('stress :', Fr.shape, F_raw[m].shape, 'match:', np.allclose(Fr, F_raw[m], atol=1e-3))

emotion: (145, 119) (145, 119) match: True
stress : (29, 154) (29, 154) match: True


In [ ]:
%%writefile /content/drive/MyDrive/EEG_Emotion_Project/app.py
import tempfile, numpy as np, pandas as pd, joblib, streamlit as st, matplotlib.pyplot as plt
from pathlib import Path
from common import *

st.set_page_config(page_title='EEG Emotion & Stress', layout='wide')
st.title('🧠 EEG-Based Emotion & Mental Stress Recognition')
st.caption('Research prototype (GAMEEMO + PhysioNet mental-arithmetic data). Not a medical device.')
MD = Path(__file__).parent / 'models'

@st.cache_resource
def load(name): return joblib.load(MD / name)

def band_table(bp):
    m = bp.mean((0, 1))
    return pd.DataFrame({'Band': list(BANDS), 'Power (µV²)': np.round(m, 2)}), m[2] / m[3]

def plane(points):
    fig, ax = plt.subplots(figsize=(5, 5)); ax.axhline(0, c='gray'); ax.axvline(0, c='gray')
    for x, y_, s in [(-.97, .9, 'Horror / stress'), (.3, .9, 'Funny / excited'), (.55, -.9, 'Calm'), (-.97, -.9, 'Boring')]:
        ax.text(x, y_, s, alpha=.6)
    for name, v, a in points:
        ax.scatter(2*v-1, 2*a-1, s=200, zorder=3)
        ax.annotate(name, (2*v-1, 2*a-1), textcoords='offset points', xytext=(8, 8), fontsize=8)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_xlabel('Valence'); ax.set_ylabel('Arousal'); ax.set_title('Valence–Arousal plane')
    return fig

mode = st.sidebar.radio('Mode', ['Emotion (game conditions)', 'Mental stress (rest vs arithmetic)'])

if mode.startswith('Emotion'):
    st.write('Upload **one** GAMEEMO raw CSV (works without calibration, lower accuracy) or **all four** '
             'recordings of the same person (adds per-person normalisation, much higher accuracy).')
    files = st.file_uploader('GAMEEMO raw EEG CSV file(s)', type=['csv'], accept_multiple_files=True)
    if files:
        feats_d, bps = {}, {}
        for f in files:
            W = emotion_windows(pd.read_csv(f))
            if len(W) == 0: st.error(f'{f.name}: no clean windows'); continue
            feats_d[f.name], bps[f.name] = feats(W, 7)
        if len(feats_d) == 4:
            allF = np.concatenate(list(feats_d.values())); mu, sd = allF.mean(0), allF.std(0) + 1e-8
            feats_d = {k: (v - mu) / sd for k, v in feats_d.items()}; model = load('emotion_rf_subjnorm.joblib')
            st.success('4 recordings: per-person normalisation applied.')
        else:
            model = load('emotion_rf_nonorm.joblib')
            st.info('No per-person normalisation: expect roughly 50% F1 (chance = 25%).')
        rows, pts = [], []
        for name, Fn in feats_d.items():
            P = model.predict_proba(Fn).mean(0)
            rows.append({'File': name, **{c: round(float(p), 3) for c, p in zip(CLASSES, P)}, 'Predicted': CLASSES[int(P.argmax())]})
            pts.append((name, P[1] + P[3], P[2] + P[3]))
        c1, c2 = st.columns(2)
        with c1:
            st.subheader('Predicted condition'); st.dataframe(pd.DataFrame(rows), hide_index=True)
            sel = st.selectbox('Show band power for', list(bps))
            tab, ab = band_table(bps[sel]); st.table(tab); st.write(f'Alpha/Beta ratio: **{ab:.2f}**')
        with c2: st.pyplot(plane(pts))

else:
    st.write('1) Upload a **rest** EDF (eyes open, at least 30 s). 2) Upload the EDF to analyse. '
             'The rest recording calibrates the model to this person.')
    rest = st.file_uploader('Rest recording (EDF)', type=['edf'], key='rest')
    test = st.file_uploader('Recording to analyse (EDF)', type=['edf'], key='test')
    if rest and test:
        def clip(f, kind):
            with tempfile.NamedTemporaryFile(suffix='.edf', delete=False) as t:
                t.write(f.getbuffer()); return stress_clip(t.name, kind)
        with st.spinner('Processing...'):
            Wr, Wt = windows(clip(rest, 'rest'), 500), windows(clip(test, 'task'), 500)
        if len(Wr) < 8 or len(Wt) == 0:
            st.error('Recordings too short or too noisy.')
        else:
            Fr, _ = feats(Wr, 8); Ft, bp = feats(Wt, 8); base = Fr[:14]
            Ftn = (Ft - base.mean(0)) / (base.std(0) + 1e-8)
            p = load('stress_rf_cal.joblib').predict_proba(Ftn)[:, 1]
            c1, c2 = st.columns(2)
            with c1:
                st.metric('P(mental stress / task load)', f'{p.mean():.2f}')
                st.subheader('Stressed' if p.mean() > .5 else 'Relaxed / at rest')
                tab, ab = band_table(bp); st.table(tab); st.write(f'Alpha/Beta ratio: **{ab:.2f}**')
            with c2:
                st.line_chart(pd.DataFrame({'P(stress) per 4 s window': p}))
    st.caption('"Stress" here means cognitive load during mental arithmetic, not emotional anxiety.')

Writing /content/drive/MyDrive/EEG_Emotion_Project/app.py


In [ ]:
!pkill -f streamlit
!pkill -f localtunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared
!streamlit run /content/drive/MyDrive/EEG_Emotion_Project/app.py --server.port 8501 --server.headless true --server.enableCORS false --server.enableXsrfProtection false &>/content/streamlit.log &
!sleep 8
!/content/cloudflared tunnel --url http://localhost:8501

2026-10-17T13:38:06Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-17T13:38:06Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-17T13:38:09Z INF +--------------------------------------------------------------------------------------------+
2026-10-17T13:38:09Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-17T13:38:09Z INF |  https://gap-amounts-adequate-pillow.trycloudflare.com